# Data Understanding
### Real Estate Machine · Graduation Project

**Goal:** understand what we actually have before touching a single model.

We do **not** clean anything here. We only *find and document* problems.
Cleaning decisions happen in notebook 02 — and each one will need a written justification.

---

### Questions this notebook answers

1. Which of the three data files is the real source of truth?
2. How many records are there, and what does each column mean?
3. Where are the missing values hiding? (hint: they are not `NaN`)
4. What is wrong with this dataset?

## 1. Imports

If any import fails, your virtual environment is not selected in VS Code.
Press `Ctrl+Shift+P` → *Python: Select Interpreter* → choose the one inside `venv`.

In [1]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

DATA = Path("..") / "Data"
print("pandas", pd.__version__)
print("numpy ", np.__version__)
print("Data folder exists:", DATA.exists())

pandas 3.0.3
numpy  2.5.1
Data folder exists: True


## 2. What files do we have?

The `Data` folder contains three files. They are **not** three copies of the same thing.
Knowing which one to trust is the single most important decision of notebook 01.

In [2]:
for f in sorted(DATA.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size/1024:10,.1f} KB")

data.csv                          514.4 KB
data.dat                        1,681.0 KB
data_clean.csv                    463.3 KB
data_clustered.csv                885.2 KB
data_raw_parsed.csv               764.1 KB
output.csv                        514.4 KB
split_indices.csv                  49.0 KB


## 3. Look at the raw file first

`data.dat` is not a CSV. It is **nested JSON** — the raw, unprocessed export.
Always look at raw text before letting pandas parse anything for you.

In [3]:
raw_text = (DATA / "data.dat").read_text()
print(raw_text[:400])

{"houses": [{"area": {"sqft_basement": 0, "sqft_above": 1340, "sqft_living/sqft_lot": "sqft_living/sqft_lot=1340\\ 7912"}, "yr_renovated": NaN, "price": 313000.0, "waterfront": 0, "floors": 1.5, "rooms": "Number of bathrooms: 1.5; Number of bedrooms: 3", "address": "18810 Densmore Ave N, Shoreline, WA 98133, USA", "date": "20140502T000000", "yr_built": 1955, "condition": 3, "view": 0}, {"area": {"


In [4]:
# JSON does not officially support NaN, so we convert it to null before parsing
#text → objects
houses = json.loads(raw_text.replace("NaN", "null"))["houses"]
print("Number of records:", len(houses))
print()
#objects → text
print(json.dumps(houses[0], indent=2))

Number of records: 4601

{
  "area": {
    "sqft_basement": 0,
    "sqft_above": 1340,
    "sqft_living/sqft_lot": "sqft_living/sqft_lot=1340\\ 7912"
  },
  "yr_renovated": null,
  "price": 313000.0,
  "waterfront": 0,
  "floors": 1.5,
  "rooms": "Number of bathrooms: 1.5; Number of bedrooms: 3",
  "address": "18810 Densmore Ave N, Shoreline, WA 98133, USA",
  "date": "20140502T000000",
  "yr_built": 1955,
  "condition": 3,
  "view": 0
}


In [5]:
print(type(houses))              # <class 'list'>
print(type(houses[0]))           # <class 'dict'>
print(type(houses[0]["area"]))   # <class 'dict'>
print(type(houses[0]["price"]))  # <class 'float'>
print(type(houses[0]["rooms"]))  # <class 'str'>   <- the one needing re

<class 'list'>
<class 'dict'>
<class 'dict'>
<class 'float'>
<class 'str'>


### The three problems in this raw file

Look carefully at the record above:

| Problem | What you see | Why it matters |
|---|---|---|
| **Two values crammed into one string** | `"sqft_living/sqft_lot": "sqft_living/sqft_lot=1340\\ 7912"` | Two separate numeric features stuck in one text field |
| **Rooms stored as a sentence** | `"Number of bathrooms: 1.5; Number of bedrooms: 3"` | Numbers buried in text |
| **Address stored as one string** | `"18810 Densmore Ave N, Shoreline, WA 98133, USA"` | Street, city, state, zip and country all merged |

Plus `yr_renovated` is `null` instead of a value, and `date` is in a compact `20140502T000000` format.

**This is what "raw data" really looks like.** The CSV in your folder is someone's *already-processed* version — we will check in section 8 whether we can trust it.

### ⚠️ A trap: the room text is not always in the same order

Never parse this with `.split()` on position. Run the cell below and see for yourself.

In [6]:
bath_first = sum(1 for h in houses if h["rooms"].startswith("Number of bathrooms"))
bed_first  = sum(1 for h in houses if h["rooms"].startswith("Number of bedrooms"))

print(f"Records starting with bathrooms : {bath_first}")
print(f"Records starting with bedrooms  : {bed_first}")
print()
for h in houses[:5]:
    print(h["rooms"])

Records starting with bathrooms : 4201
Records starting with bedrooms  : 400

Number of bathrooms: 1.5; Number of bedrooms: 3
Number of bathrooms: 2.5; Number of bedrooms: 5
Number of bathrooms: 2.0; Number of bedrooms: 3
Number of bathrooms: 2.25; Number of bedrooms: 3
Number of bedrooms: 4; Number of bathrooms: 2.5


Because the order varies, we parse **by name using a regular expression**, not by position.

`r"bedrooms:\s*([\d.]+)"` means: find the word `bedrooms:`, skip any spaces, then capture the number.

## 4. Parse the raw file into a table

In [7]:
def parse_house(h):
    """Turn one messy JSON record into a flat dictionary."""
    area = h["area"]

    # "sqft_living/sqft_lot=1340\ 7912"  ->  1340 and 7912
    m = re.search(r"=\s*(\d+)\D+(\d+)", area["sqft_living/sqft_lot"])
    sqft_living, sqft_lot = int(m.group(1)), int(m.group(2))

    # parse by NAME, never by position
    bedrooms  = float(re.search(r"bedrooms:\s*([\d.]+)",  h["rooms"]).group(1))
    bathrooms = float(re.search(r"bathrooms:\s*([\d.]+)", h["rooms"]).group(1))

    return {
        "date":          h["date"],
        "price":         h["price"],
        "bedrooms":      bedrooms,
        "bathrooms":     bathrooms,
        "sqft_living":   sqft_living,
        "sqft_lot":      sqft_lot,
        "floors":        h["floors"],
        "waterfront":    h["waterfront"],
        "view":          h["view"],
        "condition":     h["condition"],
        "sqft_above":    area["sqft_above"],
        "sqft_basement": area["sqft_basement"],
        "yr_built":      h["yr_built"],
        "yr_renovated":  h["yr_renovated"],
        "address":       h["address"],
    }


df = pd.DataFrame([parse_house(h) for h in houses])

# errors="coerce" turns anything unparseable into NaT instead of crashing.
# We WANT to see what fails rather than have pandas quietly guess for us.
df["date_raw"] = df["date"]
df["date"] = pd.to_datetime(df["date"], format="%Y%m%dT%H%M%S", errors="coerce")

print("Shape:", df.shape)
df.head()

Shape: (4601, 16)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,address,date_raw
0,2014-05-02,"313,000.00",3.00,1.50,1340,7912,1.50,0,0,3,1340,0,1955,NaN,"18810 Densmore Ave N, Shoreline, WA 98133, USA",20140502T000000
1,2014-05-02,"2,384,000.00",5.00,2.50,3650,9050,2.00,0,4,5,3370,280,1921,NaN,"709 W Blaine St, Seattle, WA 98119, USA",20140502T000000
2,2014-05-02,"342,000.00",3.00,2.00,1930,11947,1.00,0,0,4,1930,0,1966,NaN,"26206-26214 143rd Ave SE, Kent, WA 98042, USA",20140502T000000
3,2014-05-02,"420,000.00",3.00,2.25,2000,8030,1.00,0,0,4,1000,1000,1963,NaN,"857 170th Pl NE, Bellevue, WA 98008, USA",20140502T000000
4,2014-05-02,"550,000.00",4.00,2.50,1940,10500,1.00,0,0,4,1140,800,1976,NaN,"9105 170th Ave NE, Redmond, WA 98052, USA",20140502T000000


### The dates do not all parse — and that is a finding, not an accident

If you had written `pd.to_datetime(...)` without `errors="coerce"`, the notebook would have
crashed. A common reaction to that crash is to delete the `format` argument and let pandas guess. **Do not.** Look at what actually failed.

In [8]:
broken = df[df["date"].isna()]
print("Rows with an unparseable date:", len(broken))
broken[["date_raw", "price", "address"]]

Rows with an unparseable date: 2


,date_raw,price,address
4334,20140631T000000,"248,000.00","3920 153rd Ave SE, Bellevue, WA 98006, USA"
4335,23052014T000000,"505,000.00","3019 30th Ave W, Seattle, WA 98199, USA"


Two broken dates, and they are broken in two *different* ways:

| Raw value | Problem |
|---|---|
| `20140631T000000` | **June 31st does not exist.** June has 30 days. An impossible calendar date. |
| `23052014T000000` | **Wrong order.** This is day-month-year (23 May 2014), not year-month-day. |

The second one is the more dangerous kind of error. A date like `05062014` would have parsed
*silently* into the year 0506 or been misread as a different day — no error, just a wrong number
flowing quietly into your analysis.

We record both and fix them in notebook 02. Since `date` is barely usable in this project anyway
(only 10 weeks of data), the cost is low — but finding them shows you actually inspected the data.

### Split the address into usable columns

Location is almost always the strongest price driver in real estate.
Right now it is trapped inside one string, so we split it.

In [9]:
parts = df["address"].str.split(",", expand=True)
print("Number of comma-separated parts found:", parts.shape[1])
print()
print("Rows with an unexpected number of parts:")
print(df.loc[parts[4].notna(), "address"].head() if parts.shape[1] > 4 else "none")

Number of comma-separated parts found: 4

Rows with an unexpected number of parts:
none


In [10]:
# Take from the RIGHT, because street names can contain commas
split = df["address"].str.rsplit(",", n=3, expand=True)
split.columns = ["street", "city", "statezip", "country"]

for c in split.columns:
    split[c] = split[c].str.strip()

split["zipcode"] = split["statezip"].str.extract(r"(\d{5})")

df = pd.concat([df, split], axis=1)
df[["address", "street", "city", "statezip", "zipcode", "country"]].head()

,address,street,city,statezip,zipcode,country
0,"18810 Densmore Ave N, Shoreline, WA 98133, USA",18810 Densmore Ave N,Shoreline,WA 98133,98133,USA
1,"709 W Blaine St, Seattle, WA 98119, USA",709 W Blaine St,Seattle,WA 98119,98119,USA
2,"26206-26214 143rd Ave SE, Kent, WA 98042, USA",26206-26214 143rd Ave SE,Kent,WA 98042,98042,USA
3,"857 170th Pl NE, Bellevue, WA 98008, USA",857 170th Pl NE,Bellevue,WA 98008,98008,USA
4,"9105 170th Ave NE, Redmond, WA 98052, USA",9105 170th Ave NE,Redmond,WA 98052,98052,USA


## 5. Basic structure check

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4601 entries, 0 to 4600
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           4599 non-null   datetime64[us]
 1   price          4601 non-null   float64       
 2   bedrooms       4601 non-null   float64       
 3   bathrooms      4601 non-null   float64       
 4   sqft_living    4601 non-null   int64         
 5   sqft_lot       4601 non-null   int64         
 6   floors         4601 non-null   float64       
 7   waterfront     4601 non-null   int64         
 8   view           4601 non-null   int64         
 9   condition      4601 non-null   int64         
 10  sqft_above     4601 non-null   int64         
 11  sqft_basement  4601 non-null   int64         
 12  yr_built       4601 non-null   int64         
 13  yr_renovated   229 non-null    float64       
 14  address        4601 non-null   str           
 15  date_raw       4601 non-null   s

In [12]:
print("Official missing values (NaN) per column:")
print(df.isna().sum()[lambda s: s > 0])

Official missing values (NaN) per column:
date               2
yr_renovated    4372
dtype: int64


Only `yr_renovated` reports missing values. That looks clean — **and that is exactly the trap.**

In this dataset the real missing values are disguised as **zeros**. Pandas cannot detect those for you.
You have to know the domain well enough to notice that a house cannot be sold for $0, or have 0 bedrooms.

## 6. Zeros that are really missing values

In [13]:
audit = []
for col in ["price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot",
            "sqft_above", "sqft_basement", "waterfront", "view", "yr_renovated"]:
    n_zero = int((df[col] == 0).sum())
    audit.append({
        "column": col,
        "zeros": n_zero,
        "pct": f"{n_zero / len(df) * 100:.1f}%",
    })

pd.DataFrame(audit)

,column,zeros,pct
0,price,248,5.4%
1,bedrooms,2,0.0%
2,bathrooms,2,0.0%
3,sqft_living,0,0.0%
4,sqft_lot,0,0.0%
5,sqft_above,0,0.0%
6,sqft_basement,2746,59.7%
7,waterfront,4568,99.3%
8,view,4141,90.0%
9,yr_renovated,0,0.0%


**Read that table carefully — not every zero is a problem.**

| Column | Verdict |
|---|---|
| `price` = 0 | ❌ **Impossible.** A sale price of zero is not a real transaction. |
| `bedrooms` = 0 | ❌ Suspicious. Land, or a data-entry error. |
| `bathrooms` = 0 | ❌ Suspicious. |
| `sqft_basement` = 0 | ✅ **Legitimate** — the house simply has no basement. |
| `waterfront` = 0 | ✅ Legitimate — it is a 0/1 flag. |
| `view` = 0 | ✅ Legitimate — "no view" on a 0–4 scale. |
| `yr_renovated` = 0 | ⚠️ Means "never renovated", not "unknown". Needs to become a flag. |

This distinction — knowing which zeros are real and which are missing — is the heart of data processing.

In [14]:
print("Houses with price = 0:", (df["price"] == 0).sum())
print()
df[df["price"] == 0].head(5)

Houses with price = 0: 248



,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,address,date_raw,street,city,statezip,country,zipcode
4353,2014-05-02,0.00,4.00,2.50,2200,9397,2.00,0,0,3,2200,0,1987,NaN,"5214 S 292nd St, Auburn, WA 98001, USA",20140502T000000,5214 S 292nd St,Auburn,WA 98001,USA,98001
4354,2014-05-05,0.00,3.00,1.00,1340,306848,1.00,0,0,3,1340,0,1953,NaN,"17827 Mountain View Rd NE, Duvall, WA 98019, USA",20140505T000000,17827 Mountain View Rd NE,Duvall,WA 98019,USA,98019
4355,2014-05-05,0.00,3.00,1.75,1490,10125,1.00,0,0,4,1490,0,1962,NaN,"3911 S 328th St, Federal Way, WA 98001, USA",20140505T000000,3911 S 328th St,Federal Way,WA 98001,USA,98001
4356,2014-05-05,0.00,4.00,2.50,2800,5900,1.00,0,0,3,1660,1140,1963,NaN,"7052 39th Ave NE, Seattle, WA 98115, USA",20140505T000000,7052 39th Ave NE,Seattle,WA 98115,USA,98115
4357,2014-05-05,0.00,4.00,2.75,2600,5390,1.00,0,0,4,1300,1300,1960,NaN,"2120 31st Ave W, Seattle, WA 98199, USA",20140505T000000,2120 31st Ave W,Seattle,WA 98199,USA,98199


Notice these rows look **completely normal** apart from the price.
They have real addresses, real square footage, real build years.

So the price was not "wrong" — it was simply **never recorded**.
That is a missing target variable, and you cannot invent a target. This matters again in notebook 02.

## 7. Duplicates, consistency and date range

In [15]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated on address + date + price:", df.duplicated(["address", "date", "price"]).sum())
print()
df[df.duplicated(["address", "date", "price"], keep=False)].sort_values("address").head(4)

Fully duplicated rows: 1
Duplicated on address + date + price: 1



,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,address,date_raw,street,city,statezip,country,zipcode
4336,2014-05-22,"657,500.00",3.00,2.50,2670,10496,2.00,0,0,3,2670,0,1989,NaN,"1917 235th Ct NE, Sammamish, WA 98074, USA",20140522T000000,1917 235th Ct NE,Sammamish,WA 98074,USA,98074
4337,2014-05-22,"657,500.00",3.00,2.50,2670,10496,2.00,0,0,3,2670,0,1989,NaN,"1917 235th Ct NE, Sammamish, WA 98074, USA",20140522T000000,1917 235th Ct NE,Sammamish,WA 98074,USA,98074


In [16]:
# Internal consistency check: living area should equal above-ground + basement
inconsistent = df[df["sqft_living"] != df["sqft_above"] + df["sqft_basement"]]
print("Rows where sqft_living != sqft_above + sqft_basement:", len(inconsistent))
inconsistent[["sqft_living", "sqft_above", "sqft_basement", "address"]]

Rows where sqft_living != sqft_above + sqft_basement: 2


,sqft_living,sqft_above,sqft_basement,address
4338,1280,1280,1420,"746 Boylston Ave E, Seattle, WA 98102, USA"
4339,890,590,0,"701-711 26th Ave, Seattle, WA 98122, USA"


In [17]:
print("Date range :", df["date"].min().date(), "->", df["date"].max().date())
print("Span       :", (df["date"].max() - df["date"].min()).days, "days")
print()
print(df["date"].dt.to_period("M").value_counts().sort_index())

Date range : 2014-05-02 -> 2014-07-10
Span       : 69 days

date
2014-05    1768
2014-06    2179
2014-07     652
Freq: M, Name: count, dtype: int64


🚩 **Only about 10 weeks of sales, all in 2014.**

This kills any idea of a "price trend over time" analysis — there is no time span to trend over.
Do not build seasonality features. Do mention this as a limitation in your final presentation;
naming your own constraints makes you look rigorous, not weak.

## 8. Categorical consistency — the `city` column is polluted

Numeric problems are easy to spot with `.describe()`. Text problems are not.
Always check the *rare* categories, never just the common ones.

In [18]:
print("Unique city values:", df["city"].nunique())
print()
print("The 25 rarest city values:")
print(df["city"].value_counts().tail(25))

Unique city values: 62

The 25 rarest city values:
city
Yarrow Point           4
Skykomish              3
Preston                2
Milton                 2
seattle                2
Woodenville            2
Inglewood-Finn Hill    1
Snoqualmie Pass        1
Beaux Arts Village     1
Seatle                 1
Seaattle               1
Redmund                1
Belleview              1
Bellvue                1
Samamish               1
Sureline               1
Kirklund               1
Auburnt                1
Snogualmie             1
Issaguah               1
Redmonde               1
Coronation             1
auburn                 1
redmond                1
sammamish              1
Name: count, dtype: int64


Read that list slowly:

`Seatle` · `Seaattle` · `Bellvue` · `Belleview` · `Issaguah` · `Kirklund` · `Redmonde`
`Redmund` · `Samamish` · `Snogualmie` · `Sureline` · `Woodenville` · `Auburnt` · `Coronation`
plus lowercase copies: `seattle` · `auburn` · `redmond` · `sammamish`

These are **misspellings of cities that already exist in the data.** `Sureline` is Shoreline.
`Woodenville` is Woodinville. `Coronation` is Carnation.

**Why this would quietly wreck your model:** in notebook 05 you will encode `city` as a feature.
One-hot encoding treats `Seatle` as a completely different place from `Seattle` — creating a
useless column with a single row in it, while stealing that row from the real Seattle group.
Your location signal gets diluted and no error is ever raised.

### Use the zipcode to repair the city names

`zipcode` is numeric and was not corrupted, so it is our reliable anchor.
For each suspect row we ask: *what city do the other houses with this same zipcode report?*

In [19]:
suspects = ["Seatle", "Seaattle", "seattle", "Bellvue", "Belleview", "Issaguah",
            "Kirklund", "Redmonde", "Redmund", "Samamish", "Snogualmie", "Sureline",
            "Woodenville", "Auburnt", "auburn", "redmond", "sammamish", "Coronation"]

clean_rows = df[~df["city"].isin(suspects)]
zip_to_city = clean_rows.groupby("zipcode")["city"].agg(lambda s: s.mode().iat[0])

check = df[df["city"].isin(suspects)].copy()
check["suggested_city"] = check["zipcode"].map(zip_to_city)

print(f"Suspect rows: {len(check)}  |  resolvable via zipcode: {check['suggested_city'].notna().sum()}")
check[["city", "zipcode", "suggested_city", "street"]].to_string(index=False)

Suspect rows: 20  |  resolvable via zipcode: 20


'       city zipcode suggested_city                  street\n    seattle   98103        Seattle          1156 N 93rd St\nWoodenville   98077    Woodinville       22819 NE 166th St\n     Seatle   98117        Seattle         2822 NW 90th Pl\n   Seaattle   98108        Seattle        3015 S Graham St\n    Redmund   98052        Redmond        13780 NE 77th Pl\n  Belleview   98008       Bellevue         16204 SE 2nd St\n    Bellvue   98008       Bellevue        16720 NE 23rd Pl\n   Samamish   98074      Sammamish        547 240th Ave SE\n   Sureline   98155      Shoreline       14737 28th Ave NE\n   Kirklund   98034       Kirkland        8101 NE 120th St\n    Auburnt   98001         Auburn         4313 S 289th Pl\n Snogualmie   98065     Snoqualmie   9416 Templeton Ave SE\n   Issaguah   98029       Issaquah       3654 254th Ave SE\n   Redmonde   98052        Redmond       8208 138th Ave NE\nWoodenville   98072    Woodinville       15351 NE 202nd St\n Coronation   98014      Carnation     

In [20]:
print(check[["city", "zipcode", "suggested_city"]].to_string(index=False))

       city zipcode suggested_city
    seattle   98103        Seattle
Woodenville   98077    Woodinville
     Seatle   98117        Seattle
   Seaattle   98108        Seattle
    Redmund   98052        Redmond
  Belleview   98008       Bellevue
    Bellvue   98008       Bellevue
   Samamish   98074      Sammamish
   Sureline   98155      Shoreline
   Kirklund   98034       Kirkland
    Auburnt   98001         Auburn
 Snogualmie   98065     Snoqualmie
   Issaguah   98029       Issaquah
   Redmonde   98052        Redmond
Woodenville   98072    Woodinville
 Coronation   98014      Carnation
    seattle   98115        Seattle
     auburn   98092         Auburn
    redmond   98052        Redmond
  sammamish   98075      Sammamish


Every single one resolves. We apply this fix in notebook 02 — here we only prove it works.

> **Worth remembering:** when two columns describe the same thing, use the one that is
> harder to typo. Numbers beat free text.

## 9. Where do the errors actually live?

One last check, and it is revealing. Let us see *where* in the file each problem sits.

In [21]:
issues = {
    "misspelled city": df.index[df["city"].isin(suspects)],
    "unparseable date": df.index[df["date"].isna()],
    "duplicate row": df.index[df.duplicated(keep=False)],
    "sqft inconsistency": df.index[df["sqft_living"] != df["sqft_above"] + df["sqft_basement"]],
    "price = 0": df.index[df["price"] == 0],
}

for name, idx in issues.items():
    if len(idx):
        print(f"{name:20s} count={len(idx):4d}   rows {idx.min()} .. {idx.max()}")

print()
print("Total rows in file:", len(df))

misspelled city      count=  20   rows 4309 .. 4328
unparseable date     count=   2   rows 4334 .. 4335
duplicate row        count=   2   rows 4336 .. 4337
sqft inconsistency   count=   2   rows 4338 .. 4339
price = 0            count= 248   rows 4353 .. 4600

Total rows in file: 4601


**Every problem sits in the last ~300 rows.** The first ~4,300 records are clean.

That is not how naturally messy data behaves — real errors scatter randomly throughout a file.
A clean block followed by a dirty block means the problems were **deliberately appended**.

This is confirmed by the dataset's own Kaggle page ("House price prediction",
kaggle.com/datasets/shree1992/housedata), which describes it as: *"we are wrangling a large set
of property sales records stored in an unknown format and with unknown data quality issues."*
This is a data-wrangling dataset by design — not an accident, and not something specific to this project.

Two consequences for your analysis:

1. The zero-price rows are **not a random sample**. Do not describe them as "randomly missing" — say they form a contiguous block at the end of the file.
2. Anyone who loaded `data.csv` and skipped straight to modelling has silently absorbed all of
   this. 

---

## 10. Which file should we trust?

Your folder contains `data.dat`, `data.csv` and `output.csv`.
Before building anything, we check whether they agree.

In [22]:
csv_a = pd.read_csv(DATA / "data.csv")
csv_b = pd.read_csv(DATA / "output.csv")

print(f"{'data.dat (parsed)':22s} {df.shape}")
print(f"{'data.csv':22s} {csv_a.shape}")
print(f"{'output.csv':22s} {csv_b.shape}")
print()
print("Zero prices in data.dat :", int((df['price'] == 0).sum()))
print("Zero prices in data.csv :", int((csv_a['price'] == 0).sum()))

data.dat (parsed)      (4601, 21)
data.csv               (4600, 18)
output.csv             (4600, 18)

Zero prices in data.dat : 248
Zero prices in data.csv : 49


That is a very large gap. Where did the other zero prices go?
Let us match the two files row by row on address + date.

In [23]:
left = csv_a.copy()
left["address"] = (left["street"] + ", " + left["city"] + ", "
                   + left["statezip"] + ", " + left["country"])
left["daykey"] = pd.to_datetime(left["date"]).dt.strftime("%Y%m%d")

right = df.copy()
right["daykey"] = right["date"].dt.strftime("%Y%m%d")

merged = left.merge(right, on=["address", "daykey"], suffixes=("_csv", "_dat"))
changed = merged[merged["price_csv"] != merged["price_dat"]]

print("Matched rows      :", len(merged))
print("Prices that differ:", len(changed))
print("  ...where data.dat had 0 :", int((changed['price_dat'] == 0).sum()))
print()
changed.loc[changed["price_dat"] == 0, ["address", "price_dat", "price_csv"]].head(8)

Matched rows      : 4581
Prices that differ: 201
  ...where data.dat had 0 : 199



,address,price_dat,price_csv
4333,"5214 S 292nd St, Auburn, WA 98001, USA",0.00,"237,227.86"
4334,"17827 Mountain View Rd NE, Duvall, WA 98019, USA",0.00,"117,833.33"
4336,"7052 39th Ave NE, Seattle, WA 98115, USA",0.00,"744,312.50"
4340,"3401-3599 Arapahoe Pl W, Seattle, WA 98199, USA",0.00,"439,333.33"
4341,"18738-18798 49th Pl NE, Lake Forest Park, WA 9...",0.00,"280,000.00"
4344,"1249 NE 168th St, Shoreline, WA 98155, USA",0.00,"176,225.00"
4345,"579 Alpine Ridge Pl NW, Issaquah, WA 98027, USA",0.00,"500,324.00"
4346,"2821 NE 16th St, Renton, WA 98056, USA",0.00,"444,845.00"


### 🚨 The most important finding of notebook 01

Look at the replacement prices: `237,227.857143` · `117,833.333333` · `439,333.333333`

Those repeating decimals are not real sale prices. **Real houses do not sell for $237,227.857143.**
They are **group averages** — someone filled the missing prices with a mean.

**Why this is dangerous:** `price` is your regression *target*. Training a model on invented
targets means the model learns to reproduce someone else's guesswork, and your accuracy score
becomes meaningless.

| File | Verdict |
|---|---|
| `data.dat` | ✅ **Use this.** The genuine raw source. |
| `data.csv` | ⚠️ Already processed, with imputed prices baked in. Useful only for cross-checking. |
| `output.csv` | ❌ Same as `data.csv` but with corrupted `sqft_living` values. Ignore it. |

In [24]:
# Proof of the corruption in output.csv
bad = csv_b[csv_b["sqft_living"] != csv_b["sqft_above"] + csv_b["sqft_basement"]]
print("Rows in output.csv failing the sqft consistency check:", len(bad))
bad[["sqft_living", "sqft_above", "sqft_basement", "street"]]

Rows in output.csv failing the sqft consistency check: 2


,sqft_living,sqft_above,sqft_basement,street
4337,1280,1280,1420,746 Boylston Ave E
4338,890,590,0,701-711 26th Ave


## 11. First look at the numbers

In [25]:
df.describe().T[["count", "mean", "std", "min", "50%", "max"]]

,count,mean,std,min,50%,max
date,4599,2014-06-07 03:06:55.655577,NaN,2014-05-02 00:00:00,2014-06-09 00:00:00,2014-07-10 00:00:00
price,"4,601.00","534,523.98","571,601.59",0.00,"453,500.00","26,590,000.00"
bedrooms,"4,601.00",3.40,0.91,0.00,3.00,9.00
bathrooms,"4,601.00",2.16,0.78,0.00,2.25,8.00
sqft_living,"4,601.00","2,139.22",963.09,370.00,"1,980.00","13,540.00"
sqft_lot,"4,601.00","14,851.57","35,880.59",638.00,"7,683.00","1,074,218.00"
floors,"4,601.00",1.51,0.54,1.00,1.50,3.50
waterfront,"4,601.00",0.01,0.08,0.00,0.00,1.00
view,"4,601.00",0.24,0.78,0.00,0.00,4.00
condition,"4,601.00",3.45,0.68,1.00,3.00,5.00


Three things to notice, and write down:

1. **`price` mean is far above its median.** Mean ≈ 534k, median ≈ 453k. That gap means a
   strong right skew — a few very expensive houses pull the average up. This is why we will
   model `log(price)` later.
2. **Max price is $26,590,000** against a median of $453,000. Roughly 59× the typical house.
3. **`bedrooms` and `bathrooms` have a minimum of 0.** Already flagged above.

In [26]:
print("Unique cities   :", df["city"].nunique())
print("Unique zipcodes :", df["zipcode"].nunique())
print("Unique countries:", df['country'].unique(), "<- one value only, carries no information")
print()
print("Top 10 cities by number of sales:")
print(df["city"].value_counts().head(10))

Unique cities   : 62
Unique zipcodes : 77
Unique countries: <StringArray>
['USA']
Length: 1, dtype: str <- one value only, carries no information

Top 10 cities by number of sales:
city
Seattle        1569
Renton          293
Bellevue        284
Redmond         232
Kirkland        186
Issaquah        186
Kent            185
Sammamish       174
Auburn          174
Federal Way     148
Name: count, dtype: int64


## 12. Save the parsed raw data

We save the **parsed but uncleaned** table. Notebook 02 starts from here.

Keeping raw and cleaned data in separate files means you can always retrace your steps —
and show a reviewer exactly what you changed.

In [27]:
out = DATA / "data_raw_parsed.csv"
df.to_csv(out, index=False)
print("Saved:", out.resolve())
print("Shape:", df.shape)

Saved: C:\Users\MY LAP\Documents\Graduation Project\Data\data_raw_parsed.csv
Shape: (4601, 21)


---

## Data Quality Report

**Source file chosen:** `data.dat` — 4,601 raw records, nested JSON.

### Structural issues (fixed here by parsing)

| # | Issue | Resolution |
|---|---|---|
| 1 | `sqft_living` and `sqft_lot` merged into one string | Regex extraction into two numeric columns |
| 2 | Bedrooms / bathrooms stored as a sentence, in inconsistent order | Regex extraction **by name** |
| 3 | Address stored as one string | Right-split into street / city / statezip / country, zipcode extracted |
| 4 | Date in `20140502T000000` format | Parsed to datetime |

### Quality issues (decisions deferred to notebook 02)

| # | Issue | Count | Planned action |
|---|---|---|---|
| 1 | `price` = 0 — missing target | 248 | Remove. A target cannot be invented. |
| 2 | `bedrooms` or `bathrooms` = 0 | few | Inspect individually |
| 3 | Exact duplicate record | 1 | Remove |
| 4 | `sqft_living` ≠ `sqft_above` + `sqft_basement` | 2 | Inspect; likely recompute |
| 5 | `yr_renovated` null for ~95% of rows | 4,372 | Convert to `was_renovated` flag + `effective_year` |
| 6 | `country` has a single value | all | Drop the column |
| 7 | Extreme price outlier ($26.6M) | 1 | Keep, but model `log(price)` |
| 8 | Only ~10 weeks of data | all | No time-based features; state as a limitation |
| 9 | Misspelled / lowercase city names | 20 | Repair using `zipcode` as the anchor |
| 10 | Impossible date `20140631` (June 31st) | 1 | Set to NaT or correct to 2014-06-30 |
| 11 | Date in DD-MM-YYYY order `23052014` | 1 | Reorder to 2014-05-23 |

### Structural observation

All defects sit in the **last ~300 rows** (index ≈ 4,309 onward); the first ~4,300 records are
clean. The dirt was appended deliberately, so the missing prices are a contiguous block, **not**
a random sample. Describe them that way in the final report.

### Cross-file finding

`data.csv` replaced **199 missing prices with group averages**. Since `price` is the regression
target, those rows would inject fabricated ground truth into the model. We therefore build from
`data.dat` and treat `data.csv` as a reference only.